# Olist Brazilian E-Commerce → PostgreSQL

This notebook follows the supplied plan: download the Olist Brazilian E-Commerce dataset, create an explicit PostgreSQL schema, load the 8 relational CSVs with pandas + SQLAlchemy, and verify counts/data quality.

## 0. Project setup

Recommended workspace:

```text
Olist_Project/
├── .env
├── .gitignore
├── data_loader.ipynb
└── data/
    └── raw/
```

Your `.env` must use dotenv syntax (no `set`):

```text
PGHOST=localhost
PGPORT=5432
PGUSER=postgres
PGPASSWORD=YOUR_ACTUAL_POSTGRES_PASSWORD
OLIST_DB_NAME=olist
```

Kaggle credentials should be at:

```text
C:\Users\manas\.kaggle\kaggle.json
```

In [1]:
# Cell 1 - Imports and Project Paths

import os
import sys
import subprocess
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

PROJECT_DIR = Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Python version:", sys.version)
print("Project:", PROJECT_DIR)
print("Raw data:", RAW_DIR)

Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Project: c:\Users\manas\OneDrive\Desktop\Projects - AI + BI
Raw data: c:\Users\manas\OneDrive\Desktop\Projects - AI + BI\data\raw


In [2]:
# Cell 2 - Load Environment Variables and validate

env_path = find_dotenv()
if not env_path:
    raise FileNotFoundError("No .env file found. Please create one in the project root directory.")

load_dotenv(env_path)
required_env_vars = ["PGHOST", "PGPORT", "PGUSER", "PGPASSWORD", "PGDATABASE"]
missing_vars = [var for var in required_env_vars if not os.getenv(var)]
if missing_vars:
    raise EnvironmentError(f"Missing required environment variables: {', '.join(missing_vars)}")

print(".env: ", env_path)
print("PGHOST:", os.getenv("PGHOST"))
print("PGPORT:", os.getenv("PGPORT"))
print("PGUSER:", os.getenv("PGUSER"))
print("PGPASSWORD: [loaded, hidden]")
print("PGDATABASE:", os.getenv("PGDATABASE"))

.env:  c:\Users\manas\OneDrive\Desktop\Projects - AI + BI\.env
PGHOST: localhost
PGPORT: 5432
PGUSER: postgres
PGPASSWORD: [loaded, hidden]
PGDATABASE: olist


In [3]:
# Cell 3 - Verify Kaggle credentials and CLI

kaggle_json_path = Path.home() / ".kaggle" / "kaggle.json"
if not kaggle_json_path.exists():
    raise FileNotFoundError(f"Kaggle credentials not found at {kaggle_json_path}. Please ensure you have the Kaggle API key set up.")

result = subprocess.run(["kaggle", "--version"], capture_output=True, text=True)
if result.returncode != 0:
    raise EnvironmentError("Kaggle CLI is not installed or not found in PATH. Please install it and ensure it's accessible.")

print("Kaggle Credentials: Found at", kaggle_json_path)
print("Kaggle CLI Version:", result.stdout.strip())

Kaggle Credentials: Found at C:\Users\manas\.kaggle\kaggle.json
Kaggle CLI Version: Kaggle API 1.7.4.5


In [4]:
# Cell 4 - Download the dataset from Kaggle

subprocess.run(["kaggle", "datasets", "download", 
                "-d", "olistbr/brazilian-ecommerce", 
                "-p", str(RAW_DIR), "--unzip"], check=True)

print("Dataset downloaded and extracted to:", RAW_DIR)

Dataset downloaded and extracted to: c:\Users\manas\OneDrive\Desktop\Projects - AI + BI\data\raw


In [5]:
# Cell 5 - Check the contents of the raw data directory and match with expected files

expected_files = {
    "olist_customers_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
}

found = {p.name for p in RAW_DIR.glob("*.csv")}
for name in sorted(found):
    print(" -", name)

missing = sorted(expected_files - found)
if missing:
    raise FileNotFoundError("Missing files: " + ", ".join(missing))

print("All 8 load-target CSVs are present.")

 - olist_customers_dataset.csv
 - olist_geolocation_dataset.csv
 - olist_order_items_dataset.csv
 - olist_order_payments_dataset.csv
 - olist_order_reviews_dataset.csv
 - olist_orders_dataset.csv
 - olist_products_dataset.csv
 - olist_sellers_dataset.csv
 - product_category_name_translation.csv
All 8 load-target CSVs are present.


In [6]:
# Cell 6 - Inspect columns before loading into database

for path in sorted(RAW_DIR.glob("*.csv")):
    if path.name == "olist_geolocation_dataset.csv":
        continue
    sample_df = pd.read_csv(path, nrows=3)
    print("\n" + "=" * 90)
    print(path.name)
    print("Columns:")
    for col in sample_df.columns:
        print(" ", col)


olist_customers_dataset.csv
Columns:
  customer_id
  customer_unique_id
  customer_zip_code_prefix
  customer_city
  customer_state

olist_order_items_dataset.csv
Columns:
  order_id
  order_item_id
  product_id
  seller_id
  shipping_limit_date
  price
  freight_value

olist_order_payments_dataset.csv
Columns:
  order_id
  payment_sequential
  payment_type
  payment_installments
  payment_value

olist_order_reviews_dataset.csv
Columns:
  review_id
  order_id
  review_score
  review_comment_title
  review_comment_message
  review_creation_date
  review_answer_timestamp



olist_orders_dataset.csv
Columns:
  order_id
  customer_id
  order_status
  order_purchase_timestamp
  order_approved_at
  order_delivered_carrier_date
  order_delivered_customer_date
  order_estimated_delivery_date

olist_products_dataset.csv
Columns:
  product_id
  product_category_name
  product_name_lenght
  product_description_lenght
  product_photos_qty
  product_weight_g
  product_length_cm
  product_height_cm
  product_width_cm

olist_sellers_dataset.csv
Columns:
  seller_id
  seller_zip_code_prefix
  seller_city
  seller_state

product_category_name_translation.csv
Columns:
  product_category_name
  product_category_name_english


# Phase 2 — PostgreSQL

Create/connect to the `olist` database, then create explicit tables and indexes.

In [7]:
# Cell 7 - Create a connection to the PostgreSQL database

DB_NAME = os.getenv("PGDATABASE")
PG_HOST = os.getenv("PGHOST")
PG_PORT = os.getenv("PGPORT")
PG_USER = os.getenv("PGUSER")
PG_PASSWORD = os.getenv("PGPASSWORD")

def make_url(database: str) -> URL:
    return URL.create(
        drivername="postgresql+psycopg",
        username=PG_USER,
        password=PG_PASSWORD,
        host=PG_HOST,
        port=PG_PORT,
        database=database
    )

print(DB_NAME, PG_HOST, PG_PORT, PG_USER)

olist localhost 5432 postgres


In [8]:
# Cell 8 — Create database if needed
server_engine = create_engine(make_url("postgres"), future=True)

if not DB_NAME.replace("_", "").isalnum() or DB_NAME[0].isdigit():
    raise ValueError(f"Unsafe database name: {DB_NAME!r}")

with server_engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
    exists = conn.execute(
        text("SELECT 1 FROM pg_database WHERE datname = :name"),
        {"name": DB_NAME},
    ).scalar()
    if exists:
        print(f"Database '{DB_NAME}' already exists.")
    else:
        conn.exec_driver_sql(f'CREATE DATABASE "{DB_NAME}"')
        print(f"Database '{DB_NAME}' created.")
server_engine.dispose()

Database 'olist' created.


In [9]:
# Cell 9 - Connect to olist database

engine = create_engine(make_url(DB_NAME), future=True, pool_pre_ping=True)
with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar())
    print("Current DB:", conn.execute(text("SELECT current_database()")).scalar())

PostgreSQL 18.4 on x86_64-windows, compiled by msvc-19.44.35226, 64-bit
Current DB: olist


## Phase 2.1 — Explicit schema

Tables are created explicitly so pandas does not infer the database types. `DataFrame.to_sql` supports SQLAlchemy engines, `append`, `chunksize`, and `method="multi"`. citeturn170204search0

In [10]:
# Cell 10 — Create tables and indexes
SCHEMA_SQL = r"""
CREATE TABLE IF NOT EXISTS customers (
    customer_id              VARCHAR(32) PRIMARY KEY,
    customer_unique_id       VARCHAR(32) NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            VARCHAR(100),
    customer_state           CHAR(2)
);

CREATE TABLE IF NOT EXISTS sellers (
    seller_id              VARCHAR(32) PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            VARCHAR(100),
    seller_state            CHAR(2)
);

CREATE TABLE IF NOT EXISTS products (
    product_id                 VARCHAR(32) PRIMARY KEY,
    product_category_name      VARCHAR(100),
    product_name_lenght        INTEGER,
    product_description_lenght INTEGER,
    product_photos_qty         INTEGER,
    product_weight_g           NUMERIC(12,2),
    product_length_cm          NUMERIC(12,2),
    product_height_cm          NUMERIC(12,2),
    product_width_cm           NUMERIC(12,2)
);

CREATE TABLE IF NOT EXISTS product_category_name_translation (
    product_category_name         VARCHAR(100) PRIMARY KEY,
    product_category_name_english VARCHAR(100)
);

CREATE TABLE IF NOT EXISTS orders (
    order_id                       VARCHAR(32) PRIMARY KEY,
    customer_id                    VARCHAR(32) NOT NULL,
    order_status                   VARCHAR(30),
    order_purchase_timestamp       TIMESTAMP,
    order_approved_at              TIMESTAMP,
    order_delivered_carrier_date   TIMESTAMP,
    order_delivered_customer_date  TIMESTAMP,
    order_estimated_delivery_date  TIMESTAMP
);

CREATE TABLE IF NOT EXISTS order_items (
    order_id            VARCHAR(32) NOT NULL,
    order_item_id       INTEGER NOT NULL,
    product_id          VARCHAR(32),
    seller_id            VARCHAR(32),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12,2),
    freight_value       NUMERIC(12,2),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE TABLE IF NOT EXISTS order_payments (
    order_id               VARCHAR(32) NOT NULL,
    payment_sequential      INTEGER NOT NULL,
    payment_type            VARCHAR(30),
    payment_installments   INTEGER,
    payment_value          NUMERIC(12,2),
    PRIMARY KEY (order_id, payment_sequential)
);

CREATE TABLE IF NOT EXISTS order_reviews (
            review_id               VARCHAR(32) NOT NULL,
            order_id                VARCHAR(32) NOT NULL,
            review_score            INTEGER,
            review_comment_title    TEXT,
            review_comment_message  TEXT,
            review_creation_date    TIMESTAMP,
            review_answer_timestamp TIMESTAMP,
            PRIMARY KEY (review_id, order_id)
        );

CREATE INDEX IF NOT EXISTS idx_orders_customer_id ON orders(customer_id);
CREATE INDEX IF NOT EXISTS idx_order_items_product_id ON order_items(product_id);
CREATE INDEX IF NOT EXISTS idx_order_items_seller_id ON order_items(seller_id);
CREATE INDEX IF NOT EXISTS idx_order_reviews_order_id ON order_reviews(order_id);
CREATE INDEX IF NOT EXISTS idx_products_category_name ON products(product_category_name);
CREATE INDEX IF NOT EXISTS idx_customers_unique_id ON customers(customer_unique_id);
"""

with engine.begin() as conn:
    conn.exec_driver_sql(SCHEMA_SQL)

print("Tables and indexes created.")

Tables and indexes created.


In [11]:
# Cell 11 - Check the tables in the database

TABLES = ['customers', 'sellers', 'products', 
          'product_category_name_translation', 'orders', 
          'order_items', 'order_payments', 'order_reviews']

with engine.connect() as conn:
    for table in TABLES:
        exists = conn.execute(text("""
            SELECT EXISTS (
                SELECT 1 FROM information_schema.tables
                WHERE table_schema = 'public' AND table_name = :table_name
            )
        """), {"table_name": table}).scalar_one()
        print(f"{table:40} {'OK' if exists else 'MISSING'}")

customers                                OK
sellers                                  OK
products                                 OK
product_category_name_translation        OK
orders                                   OK
order_items                              OK
order_payments                           OK
order_reviews                            OK


In [12]:
# Cell 12 - File mapping and explicit rules

TABLE_FILES = {
    "customers": RAW_DIR / "olist_customers_dataset.csv",
    "sellers": RAW_DIR / "olist_sellers_dataset.csv",
    "products": RAW_DIR / "olist_products_dataset.csv",
    "product_category_name_translation": RAW_DIR / "product_category_name_translation.csv",
    "orders": RAW_DIR / "olist_orders_dataset.csv",
    "order_items": RAW_DIR / "olist_order_items_dataset.csv",
    "order_payments": RAW_DIR / "olist_order_payments_dataset.csv",
    "order_reviews": RAW_DIR / "olist_order_reviews_dataset.csv"
}

DATE_COLUMNS = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"]
}

INTEGER_COLUMNS = {
    "customers": ["customer_zip_code_prefix"],
    "sellers": ["seller_zip_code_prefix"],
    "products": ["product_name_lenght", "product_description_lenght", "product_photos_qty"],
    "order_items": ["order_item_id"],
    "order_payments": ["payment_sequential", "payment_installments"],
    "order_reviews": ["review_score"]
}

LOAD_ORDER = [
    "customers",
    "sellers",
    "products",
    "product_category_name_translation",
    "orders",
    "order_items",
    "order_payments",
    "order_reviews"
]

In [13]:
# Cell 13 - Clean and normalize dataframes before loading into database

def clean_dataframe(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].replace({"": pd.NA, " ": pd.NA})

    for col in DATE_COLUMNS.get(table_name, []):
        df[col] = pd.to_datetime(df[col], errors='coerce')

    for col in INTEGER_COLUMNS.get(table_name, []):
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    return df

def safe_chunksize(num_columns: int) -> int:
    # Keep near the planned 10,000 rows but leave headroom below
    # PostgreSQL's per-statement bind-parameter limit.
    return max(1, min(10_000, 60_000 // max(num_columns, 1)))

In [14]:
# Cell 14 - Load one table

def load_table(table_name: str) -> int:
    path = TABLE_FILES[table_name]
    if not path.exists():
        raise FileNotFoundError(f"CSV file for table '{table_name}' not found at {path}")

    print("\n" + "=" * 90)
    print(f"Loading table: {table_name}")

    df = pd.read_csv(path, 
                     parse_dates=DATE_COLUMNS.get(table_name, []), 
                     low_memory=False)
    df = clean_dataframe(df, table_name)
    chunksize = safe_chunksize(len(df.columns))

    print("Rows:", f"{len(df):,}")
    print("Columns:", len(df.columns))
    print("Chunksize:", f"{chunksize:,}")

    with engine.begin() as conn:
        df.to_sql(table_name, 
                  con=conn, 
                  if_exists='append', 
                  index=False, 
                  chunksize=chunksize, 
                  method='multi')

    print(f"Table '{table_name}' loaded successfully.")
    return len(df)

    

In [15]:
# Helper — count data rows in a CSV (excluding header)

def csv_row_count(path):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return max(sum(1 for _ in f) - 1, 0)

# Test

for table_name, csv_path in TABLE_FILES.items():
    print(
        f"{table_name:40} "
        f"{csv_row_count(csv_path):,} rows"
    )

customers                                99,441 rows
sellers                                  3,095 rows
products                                 32,951 rows
product_category_name_translation        71 rows
orders                                   99,441 rows


order_items                              112,650 rows
order_payments                           103,886 rows
order_reviews                            104,719 rows


In [16]:
# Cell 15 — Load all 8 tables with "already loaded" detection

loaded_counts = {}

for table_name in LOAD_ORDER:
    csv_path = TABLE_FILES[table_name]

    # Number of rows in the source CSV
    source_count = csv_row_count(csv_path)

    # Check current PostgreSQL row count
    with engine.connect() as conn:
        db_count = conn.execute(
            text(f'SELECT COUNT(*) FROM "{table_name}"')
        ).scalar_one()

    if db_count == source_count:
        # Data already exists and appears complete.
        loaded_counts[table_name] = db_count

        print(
            f"[SKIP] {table_name} data has already been loaded "
            f"({db_count:,} rows)."
        )

    else:
        # Existing data is incomplete or table is empty.
        print(
            f"[LOAD] {table_name}: "
            f"database={db_count:,}, source={source_count:,}"
        )

        loaded_counts[table_name] = load_table(table_name)

        print(
            f"[OK] {table_name} data has been loaded."
        )


print("\nLOAD SUMMARY")

for table_name, count in loaded_counts.items():
    print(f"{table_name:40} {count:>12,}")

[LOAD] customers: database=0, source=99,441

Loading table: customers


C:\Users\manas\AppData\Local\Temp\ipykernel_18764\3026664881.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


Rows: 99,441
Columns: 5
Chunksize: 10,000
Table 'customers' loaded successfully.
[OK] customers data has been loaded.
[LOAD] sellers: database=0, source=3,095

Loading table: sellers
Rows: 3,095
Columns: 4
Chunksize: 10,000


C:\Users\manas\AppData\Local\Temp\ipykernel_18764\3026664881.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


Table 'sellers' loaded successfully.
[OK] sellers data has been loaded.
[LOAD] products: database=0, source=32,951

Loading table: products


...

Rows: 112,650
Columns: 7
Chunksize: 8,571
Table 'order_items' loaded successfully.
[OK] order_items data has been loaded.
[LOAD] order_payments: database=0, source=103,886

Loading table: order_payments


C:\Users\manas\AppData\Local\Temp\ipykernel_18764\3026664881.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


Rows: 103,886
Columns: 5
Chunksize: 10,000
Table 'order_payments' loaded successfully.
[OK] order_payments data has been loaded.
[LOAD] order_reviews: database=0, source=104,719

Loading table: order_reviews


C:\Users\manas\AppData\Local\Temp\ipykernel_18764\3026664881.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


Rows: 99,224
Columns: 7
Chunksize: 8,571
Table 'order_reviews' loaded successfully.
[OK] order_reviews data has been loaded.

LOAD SUMMARY
customers                                      99,441
sellers                                         3,095
products                                       32,951
product_category_name_translation                  71
orders                                         99,441
order_items                                   112,650
order_payments                                103,886
order_reviews                                  99,224


# Phase 2.2 — Verify row counts

The supplied plan gives anchor values of approximately 99,441 orders, 112,650 order items, 99,441 customers, and 99,224 reviews. We also compare every database table to its source CSV.

In [22]:
# Cell 16 — Compare CSV row counts with PostgreSQL

def csv_row_nums(path: Path) -> int:
    """
    Count logical CSV records, correctly handling quoted fields
    that contain embedded newlines.
    """
    import csv

    with path.open(
        "r",
        encoding="utf-8",
        newline=""
    ) as f:
        reader = csv.reader(f)
        next(reader, None)  # skip header

        return sum(1 for _ in reader)

source_counter = {t: csv_row_nums(p) for t, p in TABLE_FILES.items()}
db_counts = {}

with engine.connect() as conn:
    for table in TABLE_FILES:
        db_counts[table] = conn.execute(
            text(f'SELECT COUNT(*) FROM "{table}"')
        ).scalar_one()

verification = pd.DataFrame({
    "source_csv_rows": pd.Series(source_counter),
    "postgres_rows": pd.Series(db_counts),
})
verification["difference"] = verification["postgres_rows"] - verification["source_csv_rows"]
verification["status"] = verification["difference"].eq(0).map({True: "OK", False: "MISMATCH"})
verification

,source_csv_rows,postgres_rows,difference,status
customers,99441,99441,0,OK
sellers,3095,3095,0,OK
products,32951,32951,0,OK
product_category_name_translation,71,71,0,OK
orders,99441,99441,0,OK
order_items,112650,112650,0,OK
order_payments,103886,103886,0,OK
order_reviews,99224,99224,0,OK


In [23]:
# Cell 17 — Check plan anchor counts
EXPECTED_COUNTS = {
    "customers": 99_441,
    "orders": 99_441,
    "order_items": 112_650,
    "order_reviews": 99_224,
}

for table, expected in EXPECTED_COUNTS.items():
    actual = db_counts[table]
    status = "OK" if actual == expected else "CHECK"
    print(f"[{status}] {table}: actual={actual:,}, expected={expected:,}")

[OK] customers: actual=99,441, expected=99,441
[OK] orders: actual=99,441, expected=99,441
[OK] order_items: actual=112,650, expected=112,650
[OK] order_reviews: actual=99,224, expected=99,224


# Phase 2.3 — Data quality checks

Undelivered orders are expected to have NULL `order_delivered_customer_date`. We check this separately from anomalies.

In [24]:
# Cell 18 — Core quality checks

QUALITY_SQL = {
    "undelivered_orders": """
        SELECT COUNT(*) FROM orders
        WHERE order_delivered_customer_date IS NULL
    """,
    "invalid_delivery_timestamps": """
        SELECT COUNT(*) FROM orders
        WHERE order_delivered_customer_date IS NOT NULL
          AND order_purchase_timestamp IS NOT NULL
          AND order_delivered_customer_date < order_purchase_timestamp
    """,
    "orphan_orders": """
        SELECT COUNT(*) FROM orders o
        LEFT JOIN customers c ON c.customer_id = o.customer_id
        WHERE c.customer_id IS NULL
    """,
    "orphan_products": """
        SELECT COUNT(*) FROM order_items oi
        LEFT JOIN products p ON p.product_id = oi.product_id
        WHERE oi.product_id IS NOT NULL AND p.product_id IS NULL
    """,
    "orphan_sellers": """
        SELECT COUNT(*) FROM order_items oi
        LEFT JOIN sellers s ON s.seller_id = oi.seller_id
        WHERE oi.seller_id IS NOT NULL AND s.seller_id IS NULL
    """,
}

quality_results = {}
with engine.connect() as conn:
    for name, sql in QUALITY_SQL.items():
        quality_results[name] = conn.execute(text(sql)).scalar_one()

pd.Series(quality_results, name="count").to_frame()

,count
undelivered_orders,2965
invalid_delivery_timestamps,0
orphan_orders,0
orphan_products,0
orphan_sellers,0


In [25]:
# Cell 19 — Duplicate-key checks

duplicate_checks = {
    "customer_id": """
        SELECT customer_id, COUNT(*) AS n FROM customers
        GROUP BY customer_id HAVING COUNT(*) > 1 LIMIT 10
    """,
    "order_id": """
        SELECT order_id, COUNT(*) AS n FROM orders
        GROUP BY order_id HAVING COUNT(*) > 1 LIMIT 10
    """,
    "order_item_key": """
        SELECT order_id, order_item_id, COUNT(*) AS n FROM order_items
        GROUP BY order_id, order_item_id HAVING COUNT(*) > 1 LIMIT 10
    """,
    "payment_key": """
        SELECT order_id, payment_sequential, COUNT(*) AS n FROM order_payments
        GROUP BY order_id, payment_sequential HAVING COUNT(*) > 1 LIMIT 10
    """,
}

with engine.connect() as conn:
    for name, sql in duplicate_checks.items():
        rows = conn.execute(text(sql)).fetchall()
        print(f"{name}: {len(rows)} duplicate groups")
        for row in rows:
            print(" ", row)

customer_id: 0 duplicate groups
order_id: 0 duplicate groups
order_item_key: 0 duplicate groups
payment_key: 0 duplicate groups
